# N-ATLAS on free Colab (for Telnora N-ATLAS Kit)

Runs **NCAIR1/N-ATLaS** (4-bit, fits a free T4) and **NCAIR1/NigerianAccentedEnglish** (speech-to-text) behind an
OpenAI-compatible API, exposed through a free Cloudflare tunnel.

1. Runtime -> Change runtime type -> **T4 GPU**
2. Run all cells top to bottom (first run downloads ~16 GB of weights, allow 10-20 min)
3. Copy the **Base URL** and **API key** printed at the end into the playground: https://telnora-natlas-kit.vercel.app/playground

Notes: free sessions disconnect after a few hours or when idle; rerun to get a new URL. 4-bit quantisation slightly
changes output quality versus the full model. Licence: N-ATLaS Open-Source Research and Innovation License (credit Awarri
Technologies and the Federal Ministry of Communications, Innovation and Digital Economy).

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes fastapi "uvicorn[standard]" python-multipart librosa requests

In [ ]:
# Optional: only needed if the model repo asks for a Hugging Face token. Leave blank to skip.
import os, getpass
tok = getpass.getpass("HF token (optional, press Enter to skip): ").strip()
if tok:
    os.environ["HF_TOKEN"] = tok


In [ ]:
!curl -sSL -o colab_server.py https://raw.githubusercontent.com/cutewizzy11/telnora-natlas-kit/main/deploy/colab_server.py && wc -l colab_server.py

In [ ]:
import os, secrets, subprocess, time, requests
os.environ["NATLAS_API_KEY"] = secrets.token_urlsafe(24)
log = open("server.log", "w")
server = subprocess.Popen(["uvicorn", "colab_server:app", "--host", "0.0.0.0", "--port", "8000"],
                          stdout=log, stderr=subprocess.STDOUT, env=os.environ)
for i in range(240):  # up to ~20 min for first download
    try:
        if requests.get("http://localhost:8000/health", timeout=2).ok:
            print("server ready"); break
    except Exception:
        pass
    if server.poll() is not None:
        print(open("server.log").read()[-3000:]); raise SystemExit("server exited, see log above")
    time.sleep(5)
else:
    print(open("server.log").read()[-3000:]); raise SystemExit("timed out waiting for server")


In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

In [ ]:
import re
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("Base URL :", url + "/v1")
print("API key  :", os.environ["NATLAS_API_KEY"])
print("ASR Base URL is the same as the Base URL.")


In [ ]:
# Smoke test through the tunnel
r = requests.post(url + "/v1/chat/completions",
    headers={"Authorization": "Bearer " + os.environ["NATLAS_API_KEY"]},
    json={"messages": [{"role": "user", "content": "Say hello in one short sentence."}], "max_tokens": 40}, timeout=300)
print(r.status_code, r.json()["choices"][0]["message"]["content"])
